In [1]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
torch.set_num_threads(2)

print("PyTorch:", torch.__version__)

names = ["unused", "A", "B", "x", "y", "z", "p", "q"]

def decode(row):
    return " ".join(names[int(token)] for token in row)

print(decode(torch.tensor([3, 1, 4, 2, 5, 6])))

PyTorch: 2.15.0+cpu
x A y B z p


I imported PyTorch and set up the tokens. The numbers represent different tokens, and the output shows x A y B z p. A comes before B.

In [2]:
weights = torch.tensor([0.1, 0.7, 0.2])
values = torch.tensor([[2., 0.], [0., 4.], [1., 1.]])

mixed = (weights[:, None] * values).sum(dim=0)

print("Weight sum:", weights.sum().item())
print("Mixed information:", mixed.tolist())

Weight sum: 1.0
Mixed information: [0.4000000059604645, 3.0]


Attention helps the model focus on important information. The second value has the most influence because it has the highest weight of 0.7.

In [3]:
generator = torch.Generator().manual_seed(42)
length = 6
pairs = []
seen = set()

while len(pairs) < 300:
    row = torch.randint(3, 8, (length,), generator=generator)
    positions = torch.randperm(length, generator=generator)[:2]
    left, right = sorted(positions.tolist())

    row[left], row[right] = 1, 2
    key = tuple(row.tolist())

    if key in seen:
        continue

    seen.add(key)
    partner = row.clone()
    partner[left], partner[right] = 2, 1

    pairs.append(torch.stack([row, partner]))

X_pairs = torch.stack(pairs)
y_pairs = torch.tensor([[1, 0]] * len(pairs))

for row, label in zip(X_pairs[0], y_pairs[0]):
    print(decode(row), "label", label.item())

print("Dataset shape:", X_pairs.shape)

z z y q A B label 1
z z y q B A label 0
Dataset shape: torch.Size([300, 2, 6])


I created 300 pairs of sequences. Label 1 means A comes before B, and label 0 means B comes before A. The order is what matters.

In [4]:
order = torch.randperm(len(pairs), generator=generator)

def partition(ids):
    return X_pairs[ids].reshape(-1, length), y_pairs[ids].reshape(-1)

X_train, y_train = partition(order[:180])
X_val, y_val = partition(order[180:240])
X_test, y_test = partition(order[240:])

majority_class = torch.bincount(y_train).argmax()
baseline = (y_val == majority_class).float().mean().item()

print("Sequence counts:", len(X_train), len(X_val), len(X_test))
print("Validation baseline:", baseline)

Sequence counts: 360 120 120
Validation baseline: 0.5


I split the data into training, validation, and testing sets. The baseline accuracy is 50% because both labels appear equally.

In [5]:
torch.manual_seed(7)

token_table = nn.Embedding(num_embeddings=8, embedding_dim=16)

sample = X_train[:2]
vectors = token_table(sample)

print("Token IDs:", sample.shape)
print("Token vectors:", vectors.shape)
print("A vector:", token_table(torch.tensor(1)).detach())

Token IDs: torch.Size([2, 6])
Token vectors: torch.Size([2, 6, 16])
A vector: tensor([-1.5827, -0.3246,  1.9264, -0.3300,  0.1984,  0.7821,  1.0391, -0.7245,
        -0.2093, -0.2153, -1.8157, -0.3452, -2.0615,  0.6741, -1.3233, -1.3598])


Embeddings turn tokens into numbers the model can learn from. Each token gets 16 values to represent it.

In [6]:
position_table = nn.Embedding(length, 16)
positions = torch.arange(length)

with_positions = vectors + position_table(positions)

print("Position vectors:", position_table(positions).shape)
print("Combined vectors:", with_positions.shape)

Position vectors: torch.Size([6, 16])
Combined vectors: torch.Size([2, 6, 16])


Position information helps the model understand where each token is located. This matters because A before B is different from B before A.

In [7]:
class OrderClassifier(nn.Module):
    def __init__(self, use_positions=True):
        super().__init__()

        self.use_positions = use_positions
        self.token = nn.Embedding(8, 16)
        self.position = nn.Embedding(length, 16)

        self.encoder = nn.TransformerEncoderLayer(
            d_model=16,
            nhead=2,
            dim_feedforward=32,
            dropout=0.0,
            batch_first=True,
        )

        self.output = nn.Linear(16, 2)

    def forward(self, token_ids):
        vectors = self.token(token_ids)

        if self.use_positions:
            positions = torch.arange(token_ids.size(1))
            vectors = vectors + self.position(positions)

        contextual = self.encoder(vectors)
        sequence_vector = contextual.mean(dim=1)

        return self.output(sequence_vector)


torch.manual_seed(10)

initial = OrderClassifier()

print("Output shape:", initial(X_train[:4]).shape)

Output shape: torch.Size([4, 2])


I built a transformer that uses attention to look at the tokens together. It gives two scores to help predict which token comes first.

In [8]:
without_positions = copy.deepcopy(initial)
without_positions.use_positions = False
without_positions.eval()

with torch.inference_mode():
    pair_logits = without_positions(X_val[:2])

print("Partner scores:", pair_logits)

print("Same scores:", torch.allclose(
    pair_logits[0], pair_logits[1], atol=1e-5
))

Partner scores: tensor([[0.0007, 0.2367],
        [0.0007, 0.2367]])
Same scores: True


Without position information, the model gives almost identical scores when A and B switch places. This shows why position matters.

In [9]:
def accuracy(model, X, y):
    model.eval()

    with torch.inference_mode():
        predictions = model(X).argmax(dim=1)

    return (predictions == y).float().mean().item()


def train(model):
    shuffle = torch.Generator().manual_seed(99)

    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=32,
        shuffle=True,
        generator=shuffle
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_function = nn.CrossEntropyLoss()

    best_accuracy = -1.0
    best_weights = None

    for epoch in range(50):
        model.train()

        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = loss_function(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        score = accuracy(model, X_val, y_val)

        if score > best_accuracy:
            best_accuracy = score
            best_weights = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 10 == 0:
            print("Epoch", epoch + 1, "validation", round(score, 3))

    model.load_state_dict(best_weights)

    return model

I created a training loop that runs for 50 epochs. The model learns by calculating its mistakes and updating its weights to improve accuracy.

In [10]:
full_model = train(copy.deepcopy(initial))

no_position_model = copy.deepcopy(initial)
no_position_model.use_positions = False
no_position_model = train(no_position_model)

print("Validation baseline:", baseline)
print("Without positions:", accuracy(no_position_model, X_val, y_val))
print("With positions:", accuracy(full_model, X_val, y_val))

full_model.eval()

with torch.inference_mode():
    predictions = full_model(X_val).argmax(dim=1)

for i in range(6):
    print(
        decode(X_val[i]),
        "actual", y_val[i].item(),
        "predicted", predictions[i].item()
    )

Epoch 10 validation 1.0
Epoch 20 validation 1.0
Epoch 30 validation 1.0
Epoch 40 validation 1.0
Epoch 50 validation 1.0
Epoch 10 validation 0.5
Epoch 20 validation 0.5
Epoch 30 validation 0.5
Epoch 40 validation 0.5
Epoch 50 validation 0.5
Validation baseline: 0.5
Without positions: 0.5
With positions: 1.0
x A x B p x actual 1 predicted 1
x B x A p x actual 0 predicted 0
q A B q q z actual 1 predicted 1
q B A q q z actual 0 predicted 0
A B y q p x actual 1 predicted 1
B A y q p x actual 0 predicted 0


I compared both models to see how position affects accuracy. The model with positions should perform better because it can learn the order of A and B.

In [11]:
print("Final test accuracy:", accuracy(full_model, X_test, y_test))

new_sequences = torch.tensor([
    [3, 1, 4, 2, 5, 6],
    [3, 2, 4, 1, 5, 6]
])

full_model.eval()

with torch.inference_mode():
    new_predictions = full_model(new_sequences).argmax(dim=1)

for row, prediction in zip(new_sequences, new_predictions):
    print(decode(row), "A before B?", bool(prediction.item()))

Final test accuracy: 1.0
x A y B z p A before B? True
x B y A z p A before B? False


I tested the trained model on new sequences. It predicts whether A comes before B. One limitation is that it only learned this simple rule with six tokens.

In this assignment, I learned how transformers use attention and position information to understand sequences. Attention helps tokens use information from other tokens, while position helps the model understand their order. I trained and compared two models and tested their predictions. This showed me why position information is important. The model can recognize a simple pattern, but it does not actually understand language like a chatbot.